In [10]:
import os
import pandas as pd
from datetime import datetime
import pydicom
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import csv
import re


# make sure to have dcm2niix installed
from utils.dcm2niix import *

!which dcm2niix


BERNER_DATEN_PATH='/nasdata/Berner_Daten/Other_Side_Bilddaten'
# read
ANALYSIS_PATH='/home/bule/berner_daten_analyse/data'

APP_DATA_PATH = '/home/bule/berner_daten_analyse/apps/app_manual_preselection'

/usr/local/bin/dcm2niix


In [ ]:
# Widgets
patient_dropdown = Dropdown(options=df['patient_id'].unique(), description='Patient:')
sequence_dropdown = Dropdown(description='Sequence:')
sequences_output = Output()
output = Output()

# Update sequence dropdown and display all entries for the selected patient
def update_sequences(*args):
    patient_id = patient_dropdown.value
    patient_data = df[df['patient_id'] == patient_id]
    sequence_dropdown.options = patient_data['study_id'].values

    with sequences_output:
        clear_output(wait=True)
        display(patient_data[['study_id', 'file_path', 'modality']])

# Display the selected modality visualization
def display_modality():
    clear_output(wait=True)
    patient_id = patient_dropdown.value
    study_id = sequence_dropdown.value

    if study_id is None:
        return

    row = df[(df['patient_id'] == patient_id) & (df['study_id'] == study_id)].iloc[0]
    file_path = row['file_path']
    modality = row['modality']

    # Load NIfTI file
    img = nib.load(file_path)
    data = img.get_fdata()

    def view_slice_dwi(slice_idx):
        clear_output(wait=True)
        plt.figure(figsize=(3, 3))
        plt.imshow(data[:, :, slice_idx, 1].squeeze(), cmap="gray", origin="lower")
        plt.title(f"Slice {slice_idx}")
        plt.axis("off")
        plt.show()

    def view_slice(slice_idx):
        clear_output(wait=True)
        plt.figure(figsize=(3, 3))
        plt.imshow(data[:, :, slice_idx].squeeze(), cmap="gray", origin="lower")
        plt.title(f"Slice {slice_idx}")
        plt.axis("off")
        plt.show()

    if modality == 'dwi':
        slider = IntSlider(min=0, max=data.shape[2] - 1, step=1, value=0)
        interactive_plot = interactive(view_slice_dwi, slice_idx=slider)
    elif modality == 't1':
        slider = IntSlider(min=0, max=data.shape[2] - 1, step=1, value=0)
        interactive_plot = interactive(view_slice, slice_idx=slider)

    display(interactive_plot)

# Link dropdowns and output
patient_dropdown.observe(update_sequences, names='value')
sequence_dropdown.observe(lambda change: display_modality(), names='value')

# Initial population of sequences
update_sequences()

# Layout
display(VBox([Label('Select Patient and Sequence:'), patient_dropdown, sequences_output, sequence_dropdown, output]))


### first try with manual selection neg / pos

In [ ]:
dwi_t1_all_df=pd.read_csv(os.path.join(ANALYSIS_PATH,"dwi_t1_all.csv"))
dwi_t1_all_df.sort_values(by='Patient_ID',inplace=True)
dwi_t1_all_df.head()

- sort paths

In [ ]:
NII_TARGET_PATH = '/home/bule/berner_daten_analyse/data/nifti_first_try'

source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in dwi_t1_all_df['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[0])

target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]

print(target_nii_sequences[0])

- convert all files

In [ ]:
#process_directories(source_dcm_sequences, target_nii_sequences)

### Second try with dicom_tags on image for dwi and t1 from manual selection

In [3]:
dwi_t1_df_2=pd.read_csv(os.path.join(ANALYSIS_PATH,"dwi_t1_df_2.csv"))
dwi_t1_df_2.sort_values(by='Patient_ID',inplace=True)
dwi_t1_df_2.head()

,Patient_ID,Study_ID,Sequence_ID,File_Name,Full_Path,Date,Sequence_ID_split,Modality
64,76813,20170614_4361718,ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2....,21.dcm,/nasdata/Berner_Daten/Other_Side_Bilddaten/000...,2017-06-14,ep2d_diff_M128_b0_1000_DIN,DWI
65,76813,20170614_4361718,ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2....,22.dcm,/nasdata/Berner_Daten/Other_Side_Bilddaten/000...,2017-06-14,ep2d_diff_M128_b0_1000_DIN,DWI
66,76813,20170614_4361718,ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2....,23.dcm,/nasdata/Berner_Daten/Other_Side_Bilddaten/000...,2017-06-14,ep2d_diff_M128_b0_1000_DIN,DWI
67,76813,20170614_4361718,ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2....,24.dcm,/nasdata/Berner_Daten/Other_Side_Bilddaten/000...,2017-06-14,ep2d_diff_M128_b0_1000_DIN,DWI
68,76813,20170614_4361718,ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2....,25.dcm,/nasdata/Berner_Daten/Other_Side_Bilddaten/000...,2017-06-14,ep2d_diff_M128_b0_1000_DIN,DWI


In [12]:
NII_TARGET_PATH = '/home/bule/berner_daten_analyse/data/nifti_second_try'
LOG_PATH = os.path.join(NII_TARGET_PATH, 'conversion_errors.log')

source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in dwi_t1_df_2['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[1])

target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]

print(target_nii_sequences[0])

/nasdata/Berner_Daten/Other_Side_Bilddaten/0012524182/20150113_3588770/t1_fl2d_tra__1.3.12.2.1107.5.2.36.40270.2015011320513753223269825.0.0.0
/home/bule/berner_daten_analyse/data/nifti_second_try/0013727036/20171006_4382923/t1_se_tra__1.3.12.2.1107.5.2.41.169589.2017100613315827522264852.0.0.0


In [ ]:
#process_directories(source_dcm_sequences, target_nii_sequences,log_path=LOG_PATH)# ~ 11 mins

In [ ]:
dwi_t1_df_2

# Try with own small selection

In [7]:
NII_TARGET_PATH = '/home/bule/berner_daten_analyse/data/nifti_pascal_selection_try'
LOG_PATH = os.path.join(NII_TARGET_PATH, 'conversion_errors_pascal_selection.log')



In [3]:
import json
with open(os.path.join(APP_DATA_PATH,'pascal_sequences.json')) as json_file:
    selection_pascal_dict = json.load(json_file)

rows = []
for patid, selection_dict in selection_pascal_dict.items():
    for modality, sequences in selection_dict.items():
        for sequence in sequences:
            study_id, sequence_id = sequence.split('/')
            rows.append({
                'Patient_ID': patid,
                'Study_ID': study_id,
                'Sequence_ID': sequence_id,
                'Modality': modality,
                'Labeller': 'Pascal'  
            })
selection_pascal_df = pd.DataFrame(rows)
print(selection_pascal_df)

     Patient_ID          Study_ID  \
0    0000076813  20170614_4361718   
1    0000076813  20170929_4371316   
2    0000442968  20151109_3835318   
3    0000442968  20151109_3835318   
4    0000481858  20121025_3013771   
..          ...               ...   
247  0007202016  20120911_2984878   
248  0007335466  20160906_4097302   
249  0007335466  20160906_4097302   
250  0007341989  20140116_3316435   
251  0007341989  20140116_3316435   

                                           Sequence_ID Modality Labeller  
0    ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2....      dwi   Pascal  
1    t1_se_tra__1.3.12.2.1107.5.2.41.169589.2017092...       t1   Pascal  
2    ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30...      dwi   Pascal  
3    t1_se_r_tra_KM__1.3.12.2.1107.5.2.30.27272.201...       t1   Pascal  
4    resolve_3scan_trace_tra_176_p2_TRACEW__1.3.12....      dwi   Pascal  
..                                                 ...      ...      ...  
247  t1_fl2d_tra__1.3.12.2.110

In [12]:
selection_pascal_df['Sequence_path']=selection_pascal_df.apply(lambda x: os.path.join(BERNER_DATEN_PATH,x['Patient_ID'],x['Study_ID'],x['Sequence_ID']),axis=1)
selection_pascal_df

,Patient_ID,Study_ID,Sequence_ID,Modality,Labeller,Sequence_path
0,0000076813,20170614_4361718,ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2....,dwi,Pascal,/nasdata/Berner_Daten/Other_Side_Bilddaten/000...
1,0000076813,20170929_4371316,t1_se_tra__1.3.12.2.1107.5.2.41.169589.2017092...,t1,Pascal,/nasdata/Berner_Daten/Other_Side_Bilddaten/000...
2,0000442968,20151109_3835318,ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30...,dwi,Pascal,/nasdata/Berner_Daten/Other_Side_Bilddaten/000...
3,0000442968,20151109_3835318,t1_se_r_tra_KM__1.3.12.2.1107.5.2.30.27272.201...,t1,Pascal,/nasdata/Berner_Daten/Other_Side_Bilddaten/000...
4,0000481858,20121025_3013771,resolve_3scan_trace_tra_176_p2_TRACEW__1.3.12....,dwi,Pascal,/nasdata/Berner_Daten/Other_Side_Bilddaten/000...
...,...,...,...,...,...,...
247,0007202016,20120911_2984878,t1_fl2d_tra__1.3.12.2.1107.5.2.36.40270.201209...,t1,Pascal,/nasdata/Berner_Daten/Other_Side_Bilddaten/000...
248,0007335466,20160906_4097302,ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30...,dwi,Pascal,/nasdata/Berner_Daten/Other_Side_Bilddaten/000...
249,0007335466,20160906_4097302,Tirm_tra_fs_TI_2500__1.3.12.2.1107.5.2.30.2727...,t1,Pascal,/nasdata/Berner_Daten/Other_Side_Bilddaten/000...
250,0007341989,20140116_3316435,ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30...,dwi,Pascal,/nasdata/Berner_Daten/Other_Side_Bilddaten/000...


In [13]:





source_dcm_sequences=list(selection_pascal_df['Sequence_path'])
print(source_dcm_sequences[0])

target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]
print(target_nii_sequences[0])

/nasdata/Berner_Daten/Other_Side_Bilddaten/0000076813/20170614_4361718/ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2.36.40186.2017061416562196724753041.0.0.0
/home/bule/berner_daten_analyse/data/nifti_pascal_selection_try/0000076813/20170614_4361718/ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2.36.40186.2017061416562196724753041.0.0.0


In [14]:
process_directories(source_dcm_sequences, target_nii_sequences,log_path=LOG_PATH)# ~ 11 mins

Chris Rorden's dcm2niiX version v1.0.20220505  GCC11.4.0 x86-64 (64-bit Linux)
Found 50 DICOM file(s)
Convert 50 DICOM as /home/bule/berner_daten_analyse/data/nifti_pascal_selection_try/0000076813/20170614_4361718/ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2.36.40186.2017061416562196724753041.0.0.0/ep2d_diff_M128_b0_1000_DIN_1.3.12.2.1107.5.2.36.40186.2017061416562196724753041.0.0.0_ep2d_diff_M128_b0_1000_DIN_20170614164701_6 (128x128x25x2)
Conversion required 0.294605 seconds (0.228405 for core code).
Converted: /nasdata/Berner_Daten/Other_Side_Bilddaten/0000076813/20170614_4361718/ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2.36.40186.2017061416562196724753041.0.0.0 -> /home/bule/berner_daten_analyse/data/nifti_pascal_selection_try/0000076813/20170614_4361718/ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2.36.40186.2017061416562196724753041.0.0.0
Chris Rorden's dcm2niiX version v1.0.20220505  GCC11.4.0 x86-64 (64-bit Linux)
Found 29 DICOM file(s)
Convert 29 DICOM as /home/bule/berner_

Error: Input folder invalid: b1000__1.3.46.670589.11.70452.5.0.7948.2017022110231878000
/bin/sh: 1: Syntax error: "(" unexpected


Chris Rorden's dcm2niiX version v1.0.20220505  GCC11.4.0 x86-64 (64-bit Linux)
Found 42 DICOM file(s)
Convert 42 DICOM as /home/bule/berner_daten_analyse/data/nifti_pascal_selection_try/0001615629/20170624_4371304/ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30.27272.2017062414592963388363156.0.0.0/ep2d_diff_3scan_trace_p2_1.3.12.2.1107.5.2.30.27272.2017062414592963388363156.0.0.0_ep2d_diff_3scan_trace_p2_20170624145813_2 (192x192x21x2)
Conversion required 0.465894 seconds (0.331665 for core code).
Converted: /nasdata/Berner_Daten/Other_Side_Bilddaten/0001615629/20170624_4371304/ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30.27272.2017062414592963388363156.0.0.0 -> /home/bule/berner_daten_analyse/data/nifti_pascal_selection_try/0001615629/20170624_4371304/ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30.27272.2017062414592963388363156.0.0.0
Chris Rorden's dcm2niiX version v1.0.20220505  GCC11.4.0 x86-64 (64-bit Linux)
Converted: /nasdata/Berner_Daten/Other_Side_Bilddaten/0001615629/201706

Error: Input folder invalid: mk__1.3.12.2.1107.5.2.18.41437.30000017062305304934200011876


Chris Rorden's dcm2niiX version v1.0.20220505  GCC11.4.0 x86-64 (64-bit Linux)
Found 38 DICOM file(s)
Convert 38 DICOM as /home/bule/berner_daten_analyse/data/nifti_pascal_selection_try/0001657011/20160313_3941113/ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30.27272.2016031316312397417462335.0.0.0/ep2d_diff_3scan_trace_p2_1.3.12.2.1107.5.2.30.27272.2016031316312397417462335.0.0.0_ep2d_diff_3scan_trace_p2_20160313162717_2 (192x192x19x2)
Conversion required 0.402230 seconds (0.311745 for core code).
Converted: /nasdata/Berner_Daten/Other_Side_Bilddaten/0001657011/20160313_3941113/ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30.27272.2016031316312397417462335.0.0.0 -> /home/bule/berner_daten_analyse/data/nifti_pascal_selection_try/0001657011/20160313_3941113/ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30.27272.2016031316312397417462335.0.0.0
Chris Rorden's dcm2niiX version v1.0.20220505  GCC11.4.0 x86-64 (64-bit Linux)
Found 24 DICOM file(s)
Convert 24 DICOM as /home/bule/berner_daten_anal

## Third try with selection from hakim/ Julian / martin  via browsertool 

In [ ]:
NII_TARGET_PATH = '/home/bule/berner_daten_analyse/data/nifti_third_try'
LOG_PATH = os.path.join(NII_TARGET_PATH, 'conversion_errors_3.log')

In [11]:
dwi_t1_df_3=pd.read_csv(os.path.join(ANALYSIS_PATH,"dwi_t1_df_3.csv"))
dwi_t1_df_3.sort_values(by='Patient_ID',inplace=True)
dwi_t1_df_3.head()

FileNotFoundError: [Errno 2] No such file or directory: '/home/bule/berner_daten_analyse/data/dwi_t1_df_3.csv'

In [ ]:
NII_TARGET_PATH = '/home/bule/berner_daten_analyse/data/nifti_third_try'
LOG_PATH = os.path.join(NII_TARGET_PATH, 'conversion_errors_3.log')

source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in dwi_t1_df_2['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[0])

target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]

print(target_nii_sequences[0])

In [ ]:
#process_directories(source_dcm_sequences, target_nii_sequences,log_path=LOG_PATH)# ~ 11 mins